In [0]:
%run "../libs/libs"

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

In [0]:
order_df = spark.read.format("delta").table("dev.tracking_sl.order")

In [0]:
product_df = order_df.select(col('productcategory').alias('category'),col('productsize').alias('size')\
    ,col('productvalue').alias('prouctvalue'),col('productweight').alias('weight')).distinct()

In [0]:
product_df = product_df.withColumn('ingestion_time', current_timestamp())

In [0]:
product_df.createOrReplaceTempView("temp_product")

In [0]:
%sql
merge into dev.tracking_gd.dimproduct tgt
USING ( 
  SELECT DISTINCT * 
  FROM temp_product 
) src 
on tgt.category = src.category
  and tgt.size = src.size
  and tgt.prouctvalue = src.prouctvalue
  and tgt.weight = src.weight
when not matched then
  insert (category, size, prouctvalue,weight, ingestion_time)
  Values(src.category,src.size,src.prouctvalue,src.weight,src.ingestion_time)